In [4]:
# -*- coding: utf-8 -*-
# ============================================================
# Analyse thématique (ruralité / orientalisme) avec fusion pondérée
# - Lit 4 JSON (spaCy_lg, spaCy_sm, Stanza, Camembert)
# - Compte les occurrences des listes thématiques
# - Normalise par 10k tokens par ouvrage
# - Fusion multi-modèles (pondérée: spaCy lg/sm dominants, Stanza soutien, CamemBERT léger)
# - Juge le thème global et fait une analyse diachronique (19e vs 20e)
# - Exporte CSV + table LaTeX
# ============================================================

import json, re, unicodedata
from pathlib import Path
from functools import reduce
import pandas as pd

# === 1) Listes thématiques (ajustez/complétez si besoin) ===================
ruralite_terms = {
    "lac","rivière","ruisseau","source","forêt","bois","vallon","montagne","colline",
    "pré","champ","campagne","prairie","vigne","jardin","verger","village",
    "berger","bergère","paysan","laboureur","moissonneur","oiseau","agneau","bœuf","cheval",
    "vent","orage","pluie","rosée","soleil","lune","étoiles",
    "auteuil","alpes","jura","provence","vallée du rhône"
}
orient_terms = {
    "orient","égypte","nil","caire","turquie","constantinople","byzance","perse","babylone",
    "assyrie","arabie","jérusalem","damas","bagdad","inde","ispahan","athènes","ithaque","troie",
    "sultan","vizir","calife","prophète","derviche","émir","odalisque","janissaire",
    "mosquée","minaret","palais","caravane","bazar","harem","sérail","tapis","turban",
    "désert","oasis","palmier","sable","mirage","soleil d’orient","soleil d'orient"
}

def normalize(s: str) -> str:
    """Minuscule, unifier apostrophes, enlever accents (NFKD)."""
    if s is None:
        return ""
    s = s.strip().lower().replace("’", "'")
    s = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in s if not unicodedata.combining(ch))

ruralite_norm = {normalize(x) for x in ruralite_terms}
orient_norm   = {normalize(x) for x in orient_terms}

# === 2) Chemins des 4 fichiers JSON =======================================
fichiers = {
    "spaCy_lg": "tous_entites_nommes_Spacy_lg.json",
    "spaCy_sm": "tous_entites_nommes_Spacy_sm.json",
    "Stanza": "tous_entites_nommes_Stanza.json",
    "Camembert": "tous_entites_nommes_Camembert.json",
}

# === 3) Métadonnées des ouvrages (Tokens + Siècle) ========================
meta = pd.DataFrame([
    {"Ouvrage":"Poésies et nouvelles (D’Arbouville, XIX)","Tokens":55910,"Siècle":"19e"},
    {"Ouvrage":"Poésies (Desbordes, 1820)","Tokens":31130,"Siècle":"19e"},
    {"Ouvrage":"Les Contemplations, Tome 2 (Hugo, 1856)","Tokens":66426,"Siècle":"19e"},
    {"Ouvrage":"Illuminations (Rimbaud, 1873-1875)","Tokens":20809,"Siècle":"19e"},
    {"Ouvrage":"Sagesse (Verlaine, 1881)","Tokens":14257,"Siècle":"19e"},
    {"Ouvrage":"Calligrammes (Apollinaire, 1918)","Tokens":16834,"Siècle":"20e"},
    {"Ouvrage":"Fleurs d’avril (Loiseau, XXe)","Tokens":20742,"Siècle":"20e"},
    {"Ouvrage":"Derniers vers (Noailles, XXe)","Tokens":12901,"Siècle":"20e"},
    {"Ouvrage":"Tandis que la terre tourne (Sauvage, 1910)","Tokens":20854,"Siècle":"20e"},
    {"Ouvrage":"Études et préludes (Vivien, 1901)","Tokens":39110,"Siècle":"20e"},
])

# 若 JSON 内的“ouvrage”命名与 meta 不一致，可在此映射成标准名：
alias_map = {
    "APPOLINAIRE_Caligrammes.txt": "Calligrammes (Apollinaire, 1918)",
    "DARBOUVILLE_Poesies-et-nouvelles.txt": "Poésies et nouvelles (D’Arbouville, XIX)",
    "DESBORDES-VALMORE_Poesies-1820.txt": "Poésies (Desbordes, 1820)",
    "HUGO_Contemplations-T2.txt": "Les Contemplations, Tome 2 (Hugo, 1856)",
    "LOISEAU_Fleurs-d-avril.txt": "Fleurs d’avril (Loiseau, XXe)",
    "NOAILLES_Derniers-vers.txt": "Derniers vers (Noailles, XXe)",
    "RIMBAUD_ILLUMINATIONS_ETC.txt": "Illuminations (Rimbaud, 1873-1875)",
    "SAUVAGE_Tandis-que-la-terre-tourne.txt": "Tandis que la terre tourne (Sauvage, 1910)",
    "VERLAINE_Sagesse.txt": "Sagesse (Verlaine, 1881)",
    "VIVIEN_Etudes-et-preludes.txt": "Études et préludes (Vivien, 1901)",
}


# === 4) 读取 JSON（兼容多种结构） =======================================
def read_entities(json_path: str):
    """
    输出: list[{"ouvrage": <str or None>, "entite": <str>}]
    兼容结构:
      - {"entites":[{"mot": "...", "ouvrage": "..."}]}
      - {"OuvrageA":{"entités nommées":[{"texte":"..."}]}, ...}
      - 拼接 JSON（多块对象按行或串接）
    """
    text = Path(json_path).read_text(encoding="utf-8")
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        data = []
        for m in re.finditer(r'\{.*?\}', text, flags=re.S):
            try:
                data.append(json.loads(m.group(0)))
            except Exception:
                pass

    out = []
    if isinstance(data, dict) and "entites" in data:
        # Camembert / Stanza
        for e in data["entites"]:
            mot = e.get("mot")
            ouv = e.get("ouvrage") or e.get("source")
            if mot:
                out.append({"ouvrage": alias_map.get(ouv, ouv), "entite": mot})
    elif isinstance(data, dict):
        # spaCy (par ouvrages)
        for ouv, doc in data.items():
            lst = doc.get("entités nommées") or doc.get("entités") or []
            for e in lst:
                txt = e.get("texte")
                if txt:
                    out.append({"ouvrage": alias_map.get(ouv, ouv), "entite": txt})
    elif isinstance(data, list):
        # JSON 拼接成列表的情况
        for obj in data:
            if isinstance(obj, dict) and "entites" in obj:
                for e in obj["entites"]:
                    mot = e.get("mot")
                    ouv = e.get("ouvrage")
                    if mot:
                        out.append({"ouvrage": alias_map.get(ouv, ouv), "entite": mot})
    return out

# === 5) 统计主题匹配 (par ouvrage) =======================================
def theme_counts(entities):
    rows = []
    for r in entities:
        ent = normalize(r["entite"])
        ouv = r["ouvrage"] or "UNKNOWN"
        if ent in ruralite_norm:
            rows.append({"Ouvrage": ouv, "theme": "ruralite"})
        elif ent in orient_norm:
            rows.append({"Ouvrage": ouv, "theme": "orient"})
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=["Ouvrage","ruralite","orient","total"])
    pivot = df.value_counts(["Ouvrage","theme"]).unstack(fill_value=0).reset_index()
    for col in ("ruralite","orient"):
        if col not in pivot:
            pivot[col] = 0
    pivot["total"] = pivot["ruralite"] + pivot["orient"]
    return pivot

def attach_tokens(dfm):
    out = dfm.merge(meta, on="Ouvrage", how="left")
    out["Tokens"] = out["Tokens"].fillna(1)  # 防止除零
    out["ruralite_per10k"] = out["ruralite"] / out["Tokens"] * 10000
    out["orient_per10k"]   = out["orient"]   / out["Tokens"] * 10000
    return out

# === 6) 跑四个模型，得到各自统计 =========================================
per_model = {}
for name, path in fichiers.items():
    ents = read_entities(path)
    per_model[name] = theme_counts(ents)

# 各模型总体汇总
summary = []
for name, dfm in per_model.items():
    R = int(dfm["ruralite"].sum()) if not dfm.empty else 0
    O = int(dfm["orient"].sum())   if not dfm.empty else 0
    T = R + O
    summary.append({
        "Modèle": name,
        "Ruralité": R, "Orientalisme": O, "Total": T,
        "Pct_R(%)": (R/T*100 if T else 0.0),
        "Pct_O(%)": (O/T*100 if T else 0.0)
    })
summary_df = pd.DataFrame(summary).sort_values("Total", ascending=False)
print("\n=== Résumé par modèle ===\n", summary_df)

# 按作品归一化
normed = {name: attach_tokens(dfm) for name, dfm in per_model.items()}

# 合并四模型（列名 R_* / O_*）
frames = []
for name, dfm in normed.items():
    frames.append(
        dfm[["Ouvrage","ruralite_per10k","orient_per10k"]]
        .rename(columns={
            "ruralite_per10k": f"R_{name}",
            "orient_per10k":   f"O_{name}"
        })
    )
fusion = reduce(lambda a,b: pd.merge(a,b,on="Ouvrage",how="outer"), frames).fillna(0)

# === 7) 加权融合（spaCy 双模为主，Stanza 稳定补充，CamemBERT 轻引入） ===
weights = {
    "spaCy_lg": 0.35,
    "spaCy_sm": 0.35,
    "Stanza":   0.20,
    "Camembert":0.10,
}

# 仅对存在的列加权，并做权重归一化（避免缺列导致总权重 != 1）
r_pairs = [(f"R_{m}", w) for m,w in weights.items() if f"R_{m}" in fusion.columns]
o_pairs = [(f"O_{m}", w) for m,w in weights.items() if f"O_{m}" in fusion.columns]
r_wsum  = sum(w for _,w in r_pairs) or 1.0
o_wsum  = sum(w for _,w in o_pairs) or 1.0

fusion["R_mean_per10k"] = sum(fusion[col]*w for col,w in r_pairs) / r_wsum
fusion["O_mean_per10k"] = sum(fusion[col]*w for col,w in o_pairs) / o_wsum

print("\n=== Par ouvrage (moyenne pondérée per10k) ===\n",
      fusion[["Ouvrage","R_mean_per10k","O_mean_per10k"]].sort_values("Ouvrage"))

# === 7) 整体主题判定（加权后，去掉 UNKNOWN） ==========================
fusion_with_meta = fusion.merge(meta[["Ouvrage","Siècle"]], on="Ouvrage", how="left")

# 去掉没有匹配上 Siècle 的行（即 UNKNOWN）
fusion_known = fusion_with_meta.dropna(subset=["Siècle"]).copy()

R_total = fusion_known["R_mean_per10k"].sum()
O_total = fusion_known["O_mean_per10k"].sum()
theme_global = "ruralité" if R_total > O_total else "orientalisme"
print("\n=== Thème global (pondéré, per10k, sans UNKNOWN) ===")
print({"R_total_per10k": R_total, "O_total_per10k": O_total, "Thème_global": theme_global})

# === 8) 历时性分析（19e vs 20e） =======================================
chronos = fusion_known.groupby("Siècle")[["R_mean_per10k","O_mean_per10k"]].mean().reset_index()
print("\n=== Thème par siècle (pondéré, per10k) ===\n", chronos)

for _, row in chronos.iterrows():
    century_theme = "ruralité" if row["R_mean_per10k"] > row["O_mean_per10k"] else "orientalisme"
    print(f"→ Au {row['Siècle']} siècle : Thème dominant = {century_theme}")

# === 9) 导出 CSV + LaTeX ================================================
summary_df.to_csv("theme_by_model.csv", index=False)
fusion.to_csv("theme_by_ouvrage_per10k_weighted.csv", index=False)
chronos.to_csv("theme_by_century_per10k_weighted.csv", index=False)

def to_latex_table(df):
    cols = ["Modèle","Ruralité","Orientalisme","Total","Pct_R(%)","Pct_O(%)"]
    return df[cols].to_latex(index=False, float_format="%.2f",
                             caption="Répartition des thèmes par modèle",
                             label="tab:themes_models")

latex_str = to_latex_table(summary_df)
Path("themes_models.tex").write_text(latex_str, encoding="utf-8")
print("\nLaTeX table saved to themes_models.tex")

# ================== 可选：一致性过滤（≥2 模型一致才计数） ==================
# 如需进一步抑制单模型噪声，可在主题匹配时做“二值一致性”过滤：
# 思路：把每个模型的 R_/O_ 列变成二值命中(>0 -> 1)，四列相加>=2 才认为该作品对该主题“稳定命中”，
# 然后再计算 per10k 或直接用一致性计数比较。这里不与主流程混用，按需另行实现。



=== Résumé par modèle ===
       Modèle  Ruralité  Orientalisme  Total   Pct_R(%)   Pct_O(%)
1   spaCy_sm        54            39     93  58.064516  41.935484
0   spaCy_lg        34            34     68  50.000000  50.000000
3  Camembert        34            28     62  54.838710  45.161290
2     Stanza        29            23     52  55.769231  44.230769

=== Par ouvrage (moyenne pondérée per10k) ===
                                        Ouvrage  R_mean_per10k  O_mean_per10k
0             Calligrammes (Apollinaire, 1918)       1.247475       0.831650
1                Derniers vers (Noailles, XXe)       1.356484       1.085187
2                Fleurs d’avril (Loiseau, XXe)       0.674959       0.337480
3           Illuminations (Rimbaud, 1873-1875)       1.345572       3.363929
4      Les Contemplations, Tome 2 (Hugo, 1856)       0.210761       1.106494
5                    Poésies (Desbordes, 1820)       1.236749       0.000000
6     Poésies et nouvelles (D’Arbouville, XIX)       0.